## In-Class Lecture 35-1 - Zero Findings

In Lecture 34, you made the command queue comply with Rules 1, 2, and 3. Today you will run Cobra on it, the static analyzer Gerard Holzmann built to check the Power of Ten, and work through every finding until each one is either fixed or formally documented. That is how flight software is reviewed before it is approved.

### Reviewing Opening your Environment on the Notre Dame Machines

<b>1.</b> Log into the Notre Dame Machines as you performed on Homework 1 using Tabby Terminal 

Open Tabby Terminal on your machine.

Type in: <code>ssh -Y username@student13.cse.nd.edu</code>
<ul><li>"username" is your NetID.</li>
<ul>
<li>Example: my email is matt.morrison@nd.edu, but my NetID is mmorri22, so I would enter:<p>
<code>ssh -Y mmorri22@student13.cse.nd.edu</code><p>
The <code>-Y</code> is included to treat the Notre Dame machines as a trusted client<p>
</li>
</ul>
</ul>

2. Next, we will create a directory where you will save your In-Class Code. First, we will go into the Dropbox directory where you will submit your code:

<code>cd esc-courses</code>

<code>cd fa26-cse-20133.01</code>

<code>cd dropbox</code>

Make a directory for you to practice your code for this lecture

    mkdir inclass35
    cd inclass35

Enable script so you can run the most up-to-date versions of C and C++:

    scl enable gcc-toolset-15 bash

> <b>Note:</b> If you use the <code>-std=c2x</code> and get the following error, this means you did not run the enable command<br>
> <code>gcc: error: unrecognized command line option ‘-std=c2x’; did you mean ‘-std=c1x’?</code>

To obtain the code for today's lecture, perform the following commands:

    wget https://raw.githubusercontent.com/mmorri22/cse20133/main/inclass/lec35/setup.sh
    chmod a+rx setup.sh
    ./setup.sh
    
To confirm, perform the <code>ls</code> command. You will see the following:

    burst.txt  cmd_queue.c  commands.txt  flight_assert.h  Makefile

<code>cmd_queue.c</code> is the command queue from In-Class Lecture 34 with Rules 1, 2, and 3 already applied, so everyone starts from the same working version. <code>flight_assert.h</code> contains Holzmann's <code>c_assert</code> from Reading 35-1.

### In-Class Reading Discussion

<font color = "red">Class Introduction Question #1 - Give two reasons that the standard <code>assert()</code> is not acceptable in flight software, and explain why <code>assert_ndebug_off</code> shut down the thruster on a nominal reading.</font>

<font color = "red">Class Introduction Question #2 - How is Holzmann's <code>c_assert</code> different from <code>assert()</code>? What does the function that uses it do when an assertion fails?</font>

<font color = "red">Class Introduction Question #3 - Which compiler flag in our course enforces Rule 6, and what C23 feature can enforce the first half of Rule 7?</font>

<font color = "red">Class Introduction Question #4 - Explain why <code>SQUARE(offset + 1)</code> evaluated to 7, and why <code>MAX(retries++, 2)</code> incremented <code>retries</code> twice. What should you write instead?</font>

<font color = "red">Class Introduction Question #5 - Why does Rule 9 forbid function pointers? How can flight software still choose between different BLAS implementations without them?</font>

<font color = "red">Class Introduction Question #6 - Cobra reported a Rule 3 violation in code that never calls <code>malloc</code>. Why, and what are the two acceptable ways to respond?</font>

### Step 0 - The Baseline

First confirm the program works, then run Cobra on it:

    make cmd_queue
    make burst
    make fault
    make p10

<code>make fault</code> ends with <code>Error 1</code> because the program correctly returns <code>EXIT_FAILURE</code> in safe mode. <code>make p10</code> reports five kinds of findings:

    Rule 5: counted 0 assertions; it should be > 0.84
      1: P10_3, 1 patterns :: Rule 3: do not use dynamic memory allocation after initialization
      2: P10_5, 10 patterns :: Rule 5: use minimally two assertions per function on average
      3: P10_pp, 2 patterns :: macro definitions
      4: P10_6, 1 patterns :: Limit use of global data
      5: P10_9, 1 patterns :: Rule 9: limit the use of pointers, use no more than 2 levels of dereferencing

Open <code>_P10_.txt</code> to see the line numbers for each finding. After every step, run <code>make cmd_queue</code>, <code>make burst</code>, and <code>make fault</code> again to make sure you did not break anything, then <code>make p10</code> to see what changed.

### Step 1 - Rule 3: The False Positive

You know from Reading 35-2 why Cobra reports Rule 3: the name <code>cmd_alloc</code>. Rename it to <code>cmd_acquire</code> everywhere it appears. The finding should disappear.

### Step 2 - Rule 9: The Typedef

Look at the line <code>_P10_.txt</code> reports for Rule 9. Cobra's pattern looks for a <code>typedef</code> followed by a <code>*</code> before the next semicolon, because a <code>typedef</code> that hides a pointer (like <code>typedef command *cmd_ptr;</code>) is something Rule 9 forbids.

Our <code>typedef struct queue</code> does not hide a pointer, but its first member, <code>command *head;</code>, puts a <code>*</code> before the first semicolon. Rewrite it so the <code>struct</code> is declared first and the <code>typedef</code> comes on its own line:

    struct queue {
    	command *head;
    	command *tail;
    };
    typedef struct queue queue;

Why doesn't <code>typedef struct command</code> trigger the same finding?

### Step 3 - Rule 8: Constants Without the Preprocessor

<code>P10_pp</code> counts every <code>#define</code> in a <code>.c</code> file. Replace the two <code>#define</code> constants with an enumeration, which is part of the C language instead of the preprocessor:

    enum {
    	OPCODE_LEN = 16,
    	MAX_COMMANDS = 16	/* the most commands the spacecraft can hold */
    };

Unlike a <code>#define</code>, an enumeration constant has a type and appears in the debugger. Unlike a global variable, it is not data that any function could change.

### Step 4 - Rule 6: The Global Pool

Cobra flags <code>pool</code> as global data. Any function in the file could change it, not just the four that should. Fix it by bundling the pool and its free list into a structure that <code>main</code> owns:

    typedef struct cmd_pool {
    	command nodes[MAX_COMMANDS];
    	command *free_list;
    } cmd_pool;

<ol>
    <li>Delete the global <code>pool</code> and <code>free_list</code>.</li>
    <li>Declare <code>cmd_pool pool;</code> inside <code>main</code>.</li>
    <li>Add a <code>cmd_pool *pool</code> parameter to every function that needs the pool: <code>pool_init</code>, <code>cmd_acquire</code>, <code>cmd_release</code>, <code>enqueue</code>, <code>cancel_command</code>, and <code>execute_all</code>.</li>
    <li>Inside those functions, <code>pool[i]</code> becomes <code>pool->nodes[i]</code>, and <code>free_list</code> becomes <code>pool->free_list</code>.</li>
</ol>

This is the largest step. If you run short on time, a documented deviation (Step 6) is also an acceptable professional answer. Be ready to explain which you chose and why.

### Step 5 - Rules 5 and 7: Recovery Assertions

Add <code>#include "flight_assert.h"</code> after the standard headers. Then, in every function, use <code>c_assert</code> to check the parameters, with a recovery action for each one, as in Reading 35-1:

    if (!c_assert(pool != NULL) || !c_assert(q != NULL)) {
    	return -1;
    }

Aim for at least two assertions per function on average. Watch for a new finding, <code>P10_7</code>: Cobra reports any pointer parameter that a function uses or passes along without checking it first.

Now run <code>make p10</code>. Rule 5 still says <code>counted 0 assertions</code>. Why? <b>Hint:</b> look at the exact name Cobra is counting, in the message <code>use minimally two assertions</code>, and at the name of the macro you used.

### Step 6 - The Deviation

Cobra's rule set counts only tokens spelled exactly <code>assert</code>. Holzmann's own paper names his recovery assertion <code>c_assert</code>, so his own tool's default rules do not count it. Going back to <code>assert()</code> to satisfy the tool would bring back the problems from Reading 35-1.

This is exactly what a deviation is for. Add a comment block at the top of <code>cmd_queue.c</code>:

    /*
     * DEVIATION D-1
     *   Rule:      Power of Ten Rule 5 (Cobra finding P10_5)
     *   Finding:   
     *   Why:       
     *   Evidence:  
     *   Reviewed:  
     */

For <b>Evidence</b>, show that the program does meet Rule 5. This command counts every use:

    grep -o "c_assert(" cmd_queue.c | wc -l

If you did not finish Step 4, write <b>DEVIATION D-2</b> for Rule 6 in the same format.

### Discussion

<ol>
    <li>Cobra reported five kinds of findings. How many pointed to code that was actually unsafe, and how many were patterns that only <i>looked</i> unsafe? Was it still worth changing the code for the false positives?</li>
    <li>Holzmann's own tool does not recognize Holzmann's own assertion. What does that tell you about the difference between following a rule and passing a tool?</li>
    <li>Removing the global pool added a parameter to six functions. Was the code better or worse for it? When would a deviation be the better choice?</li>
</ol>

## Compilation Run

    make cmd_queue
    make burst
    make fault
    make analyze
    make p10

When you are finished, the three tests behave exactly as they did at the start, <code>make analyze</code> reports no errors, and <code>make p10</code> reports only the finding covered by your deviation.